# Custom Configuration

> **Time:** ~15 minutes | **Level:** Intermediate

Reflexio's behavior is controlled by **configuration**:
- **Profile Extractors** — what to remember about users
- **Playbook Configs** — how to identify agent improvements
- **Tool Registration** — what tools your agent can use
- **Success Evaluation** — how to measure agent performance

### Prerequisites
- Reflexio server running (`uv run reflexio services start --only backend`)
- `OPENAI_API_KEY` set in your `.env` file

In [ ]:
import uuid

from _display_helpers import *

from reflexio import (
    AgentSuccessConfig,
    InteractionData,
    PlaybookAggregatorConfig,
    PlaybookConfig,
    ProfileExtractorConfig,
    ReflexioClient,
    ToolUseConfig,
    UserActionType,
)

# Each run uses a unique ID so the notebook is idempotent
RUN_ID = uuid.uuid4().hex[:8]

url, api_key = load_env()
client = ReflexioClient(url_endpoint=url, api_key=api_key)

## View Current Configuration

> Start by inspecting the current configuration to see what defaults are in place.

In [ ]:
config = client.get_config()
show_config(config)

# Show full JSON in collapsible
import json

collapsible(json.dumps(config.model_dump(), indent=2, default=str), summary="Full config JSON")

## Custom Profile Extractor

> A **profile extractor** tells Reflexio what to remember about users.
> You can create multiple extractors for different concerns.

In [ ]:

config.profile_extractor_configs = [
    ProfileExtractorConfig(
        extractor_name="shopping_preferences",
        profile_content_definition_prompt=(
            "Extract the customer's shopping preferences including: "
            "preferred product categories, budget range, brand preferences, "
            "and any deal-breakers or must-have features."
        ),
        context_prompt="Acme Electronics customer support conversation.",
    ),
]

In [ ]:
# Inspect the extractor we just created
extractor = config.profile_extractor_configs[0]
display(Markdown(f"**Extractor:** `{extractor.extractor_name}`"))
display(Markdown(f"**Definition:** {extractor.profile_content_definition_prompt}"))
display(Markdown(f"**Context:** {extractor.context_prompt}"))

## Multiple Extractors (Different Concerns)

> Separate extractors focus on different aspects of the user.
> This keeps each extractor's prompt focused and improves extraction quality.

In [ ]:
config.profile_extractor_configs.append(
    ProfileExtractorConfig(
        extractor_name="communication_style",
        profile_content_definition_prompt=(
            "Extract the customer's communication style: "
            "formality level, preferred contact method (email/phone/chat), "
            "timezone indicators, and any accessibility needs."
        ),
    ),
)
show_success(f"Configured {len(config.profile_extractor_configs)} profile extractors")

## Agent Playbook Configuration

> **Playbook configs** control how Reflexio identifies agent mistakes and improvements.
> The `playbook_definition_prompt` tells the LLM what patterns to look for.

In [ ]:

config.playbook_configs = [
    PlaybookConfig(
        playbook_name="quality_check",
        playbook_definition_prompt=(
            "Identify cases where the agent: gave incorrect information, "
            "missed an opportunity to use a tool, was inappropriately formal or informal, "
            "or failed to address all parts of the customer's question."
        ),
        playbook_aggregator_config=PlaybookAggregatorConfig(
            min_cluster_size=3,
        ),
    ),
]

In [ ]:
# Preview what we've configured so far
show_success(f"Profile extractors: {len(config.profile_extractor_configs)}")
for ext in config.profile_extractor_configs:
    rprint(f"  - [bold]{ext.extractor_name}[/bold]")
show_success(f"Playbook configs: {len(config.playbook_configs)}")
for pb in config.playbook_configs:
    rprint(f"  - [bold]{pb.playbook_name}[/bold] (min threshold: {pb.playbook_aggregator_config.min_cluster_size})")

## Register Agent Tools

> Tell Reflexio what **tools** your agent can use.
> This helps playbook entries reference specific tools the agent should have used.

In [ ]:

config.tool_can_use = [
    ToolUseConfig(tool_name="order_lookup", tool_description="Look up order status by order ID or customer email"),
    ToolUseConfig(tool_name="knowledge_base", tool_description="Search Acme's FAQ, return policy, and product specs"),
    ToolUseConfig(tool_name="inventory_check", tool_description="Check real-time product availability and stock levels"),
]

In [ ]:
# List registered tools
display(Markdown("### Registered Tools"))
for tool in config.tool_can_use:
    rprint(f"  [bold cyan]{tool.tool_name}[/bold cyan] — {tool.tool_description}")

## Agent Success Evaluation

> Optionally evaluate whether the agent **successfully resolved** the customer's issue.
> The `sampling_rate` controls what fraction of interactions are evaluated (1.0 = all).

In [ ]:

config.agent_success_configs = [
    AgentSuccessConfig(
        evaluation_name="resolution_quality",
        success_definition_prompt=(
            "The agent successfully resolved the customer's issue if: "
            "the customer's question was fully answered, any requested actions were completed, "
            "and the customer expressed satisfaction or the issue was objectively resolved."
        ),
        sampling_rate=1.0,
    ),
]

In [ ]:
# Preview the complete configuration before applying
show_config(config)
rprint("\n[dim]This is a local preview. Call set_config() to apply it to the server.[/dim]")

## Apply and Verify

> Push the updated configuration to the server and verify it was applied.

In [ ]:
client.set_config(config)
show_success("Configuration updated!")

# Verify
verified = client.get_config()
show_config(verified)

## Test: See Custom Extractors in Action

> Publish an interaction and see how the custom extractors work.
> With two extractors configured, Reflexio will extract both shopping preferences and communication style.

In [ ]:

USER_ID = f"config_test_{RUN_ID}"
client.publish_interaction(
    user_id=USER_ID,
    interactions=[
        InteractionData(
            role="User",
            content="I'm looking for a budget gaming laptop under $1500. I prefer ASUS or Lenovo brands.",
            user_action=UserActionType.NONE,
        ),
        InteractionData(
            role="Agent",
            content="Both have excellent options in that range! The ASUS ROG Strix at $1,399 and Lenovo Legion 5i at $1,449 are our top picks.",
            user_action=UserActionType.NONE,
        ),
        InteractionData(
            role="User",
            content="I do a lot of streaming too, so I need good thermal performance. Also, please always email me — I hate phone calls.",
            user_action=UserActionType.NONE,
        ),
        InteractionData(
            role="Agent",
            content="Both models have advanced cooling systems. The ASUS uses liquid metal thermal compound while the Lenovo has a dual-fan vapor chamber design.",
            user_action=UserActionType.NONE,
        ),
        InteractionData(
            role="User",
            content="I'll go with the ASUS. Can you check if it's in stock at the downtown store? I'd rather pick it up than wait for shipping.",
            user_action=UserActionType.NONE,
        ),
        InteractionData(
            role="Agent",
            content="The ASUS ROG Strix is available at our downtown location. I'll reserve one for you and send pickup details by email as requested.",
            user_action=UserActionType.NONE,
        ),
    ],
    source="notebook",
    session_id=f"config_session_{RUN_ID}",
    wait_for_response=True,
)
profiles = client.get_profiles(force_refresh=True, user_id=USER_ID)
show_profiles(profiles.user_profiles, title="Profiles from Custom Extractors")

In [ ]:
# Search profiles using the custom extractors
search_results = client.search_profiles(user_id=USER_ID, query="budget and brand preferences", top_k=5)
show_profiles(search_results.user_profiles, title="Search: 'budget and brand preferences'")

## Cleanup

In [ ]:
# Reset config to defaults — remove custom extractors
config.profile_extractor_configs = []
config.playbook_configs = []
config.tool_can_use = []
config.agent_success_configs = []
client.set_config(config)

client.delete_all_interactions()
client.delete_all_profiles()
show_success("Configuration reset and demo data cleaned up")

## Summary & Next Steps

You customized every aspect of Reflexio's behavior:

| Config Area | What you set |
|-------------|---------------|
| **Profile Extractors** | `shopping_preferences` + `communication_style` |
| **Playbook Config** | `quality_check` with aggregation threshold of 3 |
| **Tools** | `order_lookup`, `knowledge_base`, `inventory_check` |
| **Success Evaluation** | `resolution_quality` at 100% sampling |

### Continue Learning

| Notebook | What you'll learn |
|----------|-------------------|
| [07 — LangChain](07_langchain_integration.ipynb) | Integrate Reflexio with LangChain agents |
| [05 — Concurrent Sessions](05_concurrent_sessions.ipynb) | Multi-user load and data isolation |
| [06 — Simulation](06_real_world_simulation.ipynb) | Watch Reflexio learn from real conversations |